In [1]:
# Mount Drive + cài thư viện
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets accelerate tensorboard
!pip install -q evaluate scikit-learn albumentations

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [2]:
import os
import glob
import json
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from scipy.special import softmax as scipy_softmax

import evaluate
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from torchvision.transforms import (
    ColorJitter, Compose, GaussianBlur, RandomAdjustSharpness,
)
from transformers import (
    DefaultDataCollator,
    LayoutLMv3ForSequenceClassification,
    LayoutLMv3Processor,
    Trainer,
    TrainingArguments,
)

from transformers.trainer_utils import get_last_checkpoint

matplotlib.rcParams["figure.dpi"] = 120

In [3]:
# Cố định ngẫu nhiên (seed) và khai báo đường dẫn

SEED = 42
DRIVE_ROOT = "/content/drive/MyDrive/Document_AI"
IMAGE_DIR   = "/content/Document_AI_Project/data/data_subset"


# Đường dẫn file metadata
TRAIN_JSONL = f"{DRIVE_ROOT}/dataset/train_metadata.jsonl"
VAL_JSONL   = f"{DRIVE_ROOT}/dataset/val_metadata.jsonl"
TEST_JSONL  = f"{DRIVE_ROOT}/dataset/test_metadata.jsonl"

# Thư mục output
OUT_MAIN  = f"{DRIVE_ROOT}/outputs"
OUT_EXP   = f"{DRIVE_ROOT}/experiments"
OUT_EVAL  = f"{DRIVE_ROOT}/eval"
for d in [OUT_MAIN, OUT_EXP, OUT_EVAL]:
    os.makedirs(d, exist_ok=True)



def seed_everything(seed: int = SEED):
    """Cố định toàn bộ nguồn ngẫu nhiên để kết quả tái lập được."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed) # Cố định random của PyTorch trên CPU (Ảnh hưởng tới: khởi tạo weight, dropout, random tensor, shuffle.)
    torch.backends.cudnn.benchmark = False

seed_everything()


In [4]:
# Giải nén tập ảnh từ Drive vào Colab local

%cd /content
!mkdir -p Document_AI_Project
!unzip -q "{DRIVE_ROOT}/dataset/data_subset.zip" -d /content/Document_AI_Project/data
print("Giải nén xong: ", IMAGE_DIR)


/content
Giải nén xong:  /content/Document_AI_Project/data/data_subset


In [5]:
# Khai báo bảng nhãn (16 lớp – RVL-CDIP)
CLASS_TO_IDX = {
    "letter": 0,        "form": 1,          "email": 2,
    "handwritten": 3,   "advertisement": 4, "scientific_report": 5,
    "scientific_publication": 6,            "specification": 7,
    "file_folder": 8,   "news_article": 9,  "budget": 10,
    "invoice": 11,      "presentation": 12, "questionnaire": 13,
    "resume": 14,       "memo": 15,
}

# Danh sách nhãn theo thứ tự index (dùng cho sklearn và Trainer)
LABEL_LIST = [name for name, _ in sorted(CLASS_TO_IDX.items(), key=lambda x: x[1])]
ID2LABEL   = {i: name for i, name in enumerate(LABEL_LIST)}
LABEL2ID   = {name: i  for i, name in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

print(f"{NUM_LABELS} nhãn:")
for i, name in ID2LABEL.items():
    print(f"   {i:>2}: {name}")

16 nhãn:
    0: letter
    1: form
    2: email
    3: handwritten
    4: advertisement
    5: scientific_report
    6: scientific_publication
    7: specification
    8: file_folder
    9: news_article
   10: budget
   11: invoice
   12: presentation
   13: questionnaire
   14: resume
   15: memo


In [6]:
# Đọc file JSONL và tạo HuggingFace Dataset
# Cấu trúc mỗi dòng JSONL: { "file_name": "...", "words": [...], "bboxes": [[x0,y0,x1,y1], ...], "label": 0 }

def data_generator(jsonl_path: str):
    """Generator đọc từng dòng JSONL, bỏ qua mẫu không tìm thấy ảnh."""
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            img_path = os.path.join(IMAGE_DIR, item["file_name"])
            if not os.path.exists(img_path):
                continue
            label = item["label"]
            yield {
                "image":  img_path,
                "tokens": item["words"],
                "bboxes": item["bboxes"],
                "label":  int(label[0] if isinstance(label, list) else label),
            }

train_ds = Dataset.from_generator(data_generator, gen_kwargs={"jsonl_path": TRAIN_JSONL})
val_ds   = Dataset.from_generator(data_generator, gen_kwargs={"jsonl_path": VAL_JSONL})
test_ds  = Dataset.from_generator(data_generator, gen_kwargs={"jsonl_path": TEST_JSONL})

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 31,929 | Val: 3,991 | Test: 3,993


In [7]:
# Khởi tạo Processor và định nghĩa Transform
# LayoutLMv3Processor = ViTFeatureExtractor (ảnh) + RoBERTa Tokenizer (text)
# apply_ocr=False vì đã có token + bbox từ JSONL (không cần chạy OCR lại)


processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False,
)

# ── Augmentation chỉ áp dụng cho tập Train ─────────────────────────────────
# Chỉ dùng biến đổi màu sắc/pixel, KHÔNG dùng flip/crop/rotate
# (lý do: bbox sẽ bị lệch nếu ảnh bị cắt hay xoay)
_train_augment = Compose([
    ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2), # Thay đổi ánh sáng/độ tương phản
    GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 2.0)), # Làm mờ chữ giống scan chất lượng kém
    RandomAdjustSharpness(sharpness_factor=2, p=0.5), # Tăng độ sắc nét ngẫu nhiên
])


def _encode(examples, augment: bool = False):
    """Hàm nội bộ: mã hoá batch ảnh + text → tensor cho Trainer."""
    images = [Image.open(p).convert("RGB") for p in examples["image"]]
    if augment:
        images = [_train_augment(img) for img in images]

    #Đưa vào Processor
    enc = processor(
        images,
        examples["tokens"],
        boxes=examples["bboxes"],
        max_length=512,
        padding="max_length",
        truncation=True,
    )
    enc["labels"] = torch.tensor(examples["label"], dtype=torch.long)
    return enc


def train_transforms(examples):
    return _encode(examples, augment=True)

def eval_transforms(examples):
    return _encode(examples, augment=False)

# Kích hoạt Online Transform cho Dataset
train_ds.set_transform(train_transforms)
val_ds.set_transform(eval_transforms)
test_ds.set_transform(eval_transforms)
print("Processor và transform đã gán cho 3 tập dữ liệu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

The image processor of type `LayoutLMv3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Processor và transform đã gán cho 3 tập dữ liệu


In [8]:
# Training
"""
**Chiến lược lưu checkpoint** (tránh mất tiến trình khi hết giờ Colab):
- Cuối mỗi epoch → ghi đè `outputs/checkpoint-*` lên Drive
- Epoch tốt nhất (Macro F1 cao nhất) → `load_best_model_at_end=True`
- Khi mở lại Colab → Code chạy, tự tìm checkpoint mới nhất để tiếp tục
"""


# Hàm tính metric cho Trainer

_accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = _accuracy.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1  = f1_score(labels, preds, average="macro",  zero_division=0)
    precision = precision_score(labels, preds, average="macro", zero_division=0)
    recall    = recall_score(labels, preds, average="macro",    zero_division=0)
    return {
        "accuracy":  round(acc,       4),
        "f1":        round(macro_f1,  4),   # Trainer dùng "f1" để chọn best model
        "precision": round(precision, 4),
        "recall":    round(recall,    4),
    }
print("compute_metrics đã cập nhật (Acc + Macro F1 + P + R)")

compute_metrics đã cập nhật (Acc + Macro F1 + P + R)


In [9]:
# Load model LayoutLMv3 từ pretrained

model = LayoutLMv3ForSequenceClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)


total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {model.config.model_type} | {total_params:.1f}M params")

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForSequenceClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias           | MISSING    | 
classifier.dense.weight            | MISSING    | 
classifier.dense.bias              | MISSING    | 
classifier.out_proj.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: layoutlmv3 | 125.9M params


In [10]:
# Cấu hình huấn luyện, khởi tạo Trainer và chạy train
# Hyperparameter đã chọn:
#   lr=2e-5   : an toàn cho attention weights của Transformer
#   warmup=10%: học rate tăng dần → tránh vỡ gradient đầu epoch
#   wd=0.01 (weight decay)  : L2 regularization → giảm overfitting


training_args = TrainingArguments(
    output_dir                  = OUT_MAIN,
    logging_dir                 = f"{OUT_MAIN}/logs",
    num_train_epochs            = 5,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 2,          # Effective batch = 4×2 = 8
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    fp16                        = True,       # Mixed precision → tiết kiệm VRAM
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    logging_steps               = 50,
    save_total_limit            = 2,          # Chỉ giữ 2 checkpoint gần nhất
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    remove_unused_columns       = False,
    report_to                   = "tensorboard",
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    data_collator   = DefaultDataCollator(),
    compute_metrics = compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:

# ── Tự động tiếp tục từ checkpoint nếu có ──────────────────────────────────

last_checkpoint = get_last_checkpoint(OUT_MAIN)

if last_checkpoint is not None:
    print(f"▶ Tiếp tục training từ checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("▶ Bắt đầu training từ đầu")
    trainer.train()


▶ Tiếp tục training từ checkpoint: /content/drive/MyDrive/Document_AI/outputs/checkpoint-11976


There were missing keys in the checkpoint model loaded: ['layoutlmv3.embeddings.LayerNorm.weight', 'layoutlmv3.embeddings.LayerNorm.bias', 'layoutlmv3.LayerNorm.weight', 'layoutlmv3.LayerNorm.bias', 'layoutlmv3.encoder.layer.0.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.0.attention.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.0.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.0.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.1.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.1.attention.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.1.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.1.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.2.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.2.attention.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.2.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.2.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.3.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.3.attention.out

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
4,0.388946,0.437395,0.925600,0.925700,0.926700,0.925600
5,0.299895,0.450467,0.929800,0.929800,0.930300,0.929900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['layoutlmv3.embeddings.LayerNorm.weight', 'layoutlmv3.embeddings.LayerNorm.bias', 'layoutlmv3.LayerNorm.weight', 'layoutlmv3.LayerNorm.bias', 'layoutlmv3.encoder.layer.0.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.0.attention.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.0.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.0.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.1.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.1.attention.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.1.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.1.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.2.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.2.attention.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.2.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.2.output.LayerNorm.bias', 'layoutlmv3.encoder.layer.3.attention.output.LayerNorm.weight', 'layoutlmv3.encoder.layer.3.attention.out